# DPO Recommender System - DistillRecDial Dataset Augmentation

###Jade Webb

###Jun-Jul 2026

###Dataset: https://huggingface.co/datasets/swap-uniba/DistillRecDial

## Installations, Imports, Authentication, and Constants

In [ ]:
import os
import json
import re
import asyncio
import random
import pandas as pd
import nest_asyncio
from datasets import load_dataset
from openai import AsyncOpenAI
from tqdm.asyncio import tqdm

In [ ]:
from google.colab import userdata
from huggingface_hub import login
hf_token = userdata.get('HF_TOKEN')
if hf_token:
    login(hf_token)
else:
    print("Please set your HF_TOKEN in the Colab Secrets tab first.")
    hf_token = userdata.get('HF_TOKEN')

In [ ]:
openai_api_key = userdata.get('OPENAI_API_KEY')
if openai_api_key:
    client = AsyncOpenAI(api_key=openai_api_key)
else:
    print("Please set your OPENAI_API_KEY in the Colab Secrets tab first.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
OUTPUT_FILE = "/content/drive/MyDrive/Colab Notebooks/DPO Recommender System/DistillRecDial_DPO_Dataset.json"
OUTPUT_FILE_CLEANED = "/content/drive/MyDrive/Colab Notebooks/DPO Recommender System/DistillRecDial_DPO_Dataset_Cleaned.json"
MAX_SAMPLES = 500
CONCURRENT_REQUESTS = 5
RANDOM_STATE = 42
ASSISTANT_HEADER = "<|start_header_id|>assistant<|end_header_id|>\n"
EOT_TOKEN = "<|eot_id|>"
SPECIAL_TOKEN = "<|reserved_special_token_10|>"
SPECIAL_TOKEN_REGEX = r"<\|\s*reserved_special_token_10\s*\|>"

##Augment DistillRecDial Dataset for DPO

Isolate recommendation points in the raw dataset conversation text and use GPT-4o-mini to generate rejected responses to achieve a DPO preference structure (prompt, chosen, rejected)

In [ ]:
#Parse raw text blobs containing Llama 3 headers and identify the exact turn where a recommendation is made
#Then split into the context block (prompt) and recommendation text (chosen)
def extract_dpo_candidates(raw_dataset):

    extracted_pairs = []

    for row in raw_dataset:

        #Conversation text block
        text_content = row.get("text", "")

        #Split text block into individual assistant turns
        turns = text_content.split(ASSISTANT_HEADER)
        if len(turns) < 2:
            continue

        #Start with the system prompt and first user turn
        context = turns[0]

        #Iterate thru every assistant turn to find the recommendation turn
        for turn in turns[1:]:

            #Isolate the assistant's text response before the next user turn
            turn_parts = turn.split(EOT_TOKEN)
            assistant_text = turn_parts[0].strip()

            #Check if the assistant is discussing a movie entity
            contains_movie_token = SPECIAL_TOKEN in assistant_text

            #Check if the turn is an inquiry or clarifying question
            is_question = assistant_text.strip().endswith("?")

            #Check if the turn contains a movie recommendation (references movie and is not a question)
            #Look for recommendation keywords or the special movie token
            if contains_movie_token and not is_question:

                #Located recommendation moment
                #Cut off prompt context at the assistant header
                full_prompt = context + ASSISTANT_HEADER
                final_chosen = assistant_text + EOT_TOKEN

                extracted_pairs.append({
                    "prompt": str(full_prompt).strip(),
                    "chosen": str(final_chosen).strip()
                })

                #Stop processing and move to next row
                break

            #If not a recommendation turn, rebuild the context and move down the chat
            rebuilt_turn = turn_parts[0] + EOT_TOKEN + EOT_TOKEN.join(turn_parts[1:])
            context += ASSISTANT_HEADER + rebuilt_turn

    random.seed(RANDOM_STATE)
    random.shuffle(extracted_pairs)
    return extracted_pairs[:MAX_SAMPLES]


In [ ]:
#Calls OpenAI API asynchronously to generate a realistic faulty/unhelpful recommendation response (rejected) based on conversation history
async def generate_bad_recommendation(semaphore, context, chosen_text):

    system_instruction = (
        "You are an expert AI data-synthesis engine. Your job is to create a negative example for a DPO preference dataset.\n\n"
        "I will provide a conversation context between a User and a Recommendation Assistant, along with the correct, helpful recommendation ('chosen').\n"
        "You must generate an alternate response ('rejected') that looks realistic but is structurally BAD.\n\n"
        "Guidelines for the 'rejected' response:\n"
        "1. It must match the exact conversational style, tone, and length of the 'chosen' response.\n"
        "2. It must explicitly fail the user's intent. For example: ignore their genre constraints, recommend a movie they explicitly said they hate, "
        "or give an inaccurate, hallucinated description of an item.\n"
        "3. Do not make it cartoonishly bad; it should look like a mistake a real, flawed AI model would make.\n"
        "4. Output ONLY the response text. Do not include labels, markdown formatting like 'Rejected:', or explanations."
    )

    user_prompt = f"### CONVERSATION CONTEXT:\n{context}\n\n### CHOSEN RESPONSE:\n{chosen_text}"

    async with semaphore:
        try:
            response = await client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {"role": "system", "content": system_instruction},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=0.7,
                max_tokens=200,
                seed=RANDOM_STATE
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            print(f"\nError generating sample: {e}")
            return None

In [ ]:
async def main():
    print("Loading swap-uniba/DistillRecDial from Hugging Face...")
    raw_dataset = load_dataset("swap-uniba/DistillRecDial", split="train")

    print("Extracting state transitions using Llama tokens...")
    candidate_pairs = extract_dpo_candidates(raw_dataset)
    print(f"Extracted {len(candidate_pairs)} candidate instances for processing.")

    #Check for existing progress to allow for resuming
    processed_records = []
    existing_prompts = set()
    if os.path.exists(OUTPUT_FILE):
        with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
            try:
                processed_records = json.load(f)
                existing_prompts = {r["prompt"] for r in processed_records}
                print(f"Found existing checkpoint with {len(processed_records)} completed samples.")
            except json.JSONDecodeError:
                pass

    #Filter out already generated samples
    tasks_to_run = [p for p in candidate_pairs if p["prompt"] not in existing_prompts]
    if not tasks_to_run:
        print("All items have already been processed and saved!")
        return

    print(f"Launching async workers to generate {len(tasks_to_run)} preference variants...")
    semaphore = asyncio.Semaphore(CONCURRENT_REQUESTS)

    #Create task queue
    async_tasks = [
        generate_bad_recommendation(semaphore, item["prompt"], item["chosen"])
        for item in tasks_to_run
    ]

    #Execute tasks concurrently
    results = await tqdm.gather(*async_tasks)

    #Map back results and merge with checkpoints
    new_count = 0
    for item, rejected_text in zip(tasks_to_run, results):
        if rejected_text:
            processed_records.append({
                "prompt": item["prompt"],
                "chosen": item["chosen"],
                "rejected": rejected_text
            })
            new_count += 1

    #Save directly to disk
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(processed_records, f, indent=4, ensure_ascii=False)

    print(f"\nGeneration complete! Saved {new_count} new pairs ({len(processed_records)} total) to '{OUTPUT_FILE}'.")

In [ ]:
if __name__ == "__main__":
  nest_asyncio.apply()
  asyncio.run(main())

Loading swap-uniba/DistillRecDial from Hugging Face...


README.md:   0%|          | 0.00/4.26k [00:00<?, ?B/s]

DistillRecDial/Movies_and_TV_llama3.1_re(…):   0%|          | 0.00/72.8M [00:00<?, ?B/s]

(…).1_reserved_special_token_10_valid.jsonl:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

(…)3.1_reserved_special_token_10_test.jsonl:   0%|          | 0.00/9.04M [00:00<?, ?B/s]

...

Generating validation split:   0%|          | 0/2102 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2102 [00:00<?, ? examples/s]

Extracting state transitions using Llama tokens...
Extracted 500 candidate instances for processing.
Launching async workers to generate 500 preference variants...


100%|██████████| 500/500 [03:47<00:00,  2.20it/s]


Generation complete! Saved 500 new pairs (500 total) to '/content/drive/MyDrive/Colab Notebooks/DPO Recommender System/DistillRecDial_DPO_Dataset.json'.


##Inspect and Clean Augmented Dataset

In [ ]:
with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

In [ ]:
df = pd.DataFrame(data)

In [ ]:
df

,prompt,chosen,rejected
0,<|begin_of_text|><|start_header_id|>system<|en...,I recommend <|reserved_special_token_10|> Slee...,I suggest you check out <|reserved_special_tok...
1,<|begin_of_text|><|start_header_id|>system<|en...,"Based on what you've told me, I think I might ...",You might really enjoy <|reserved_special_toke...
2,<|begin_of_text|><|start_header_id|>system<|en...,That's a great perspective. It sounds like you...,"I see where you’re coming from, and it’s inter..."
3,<|begin_of_text|><|start_header_id|>system<|en...,The movie I'm thinking of is <|reserved_specia...,### REJECTED RESPONSE:\nThe movie I'm thinking...
4,<|begin_of_text|><|start_header_id|>system<|en...,"Mercenary movies can be a lot of fun, with the...",If you're in the mood for action and adventure...
...,...,...,...
495,<|begin_of_text|><|start_header_id|>system<|en...,Animated adventures can be very captivating. I...,You might want to check out some intense thril...
496,<|begin_of_text|><|start_header_id|>system<|en...,I was thinking of <|reserved_special_token_10|...,You might really enjoy a classic like The Emoj...
497,<|begin_of_text|><|start_header_id|>system<|en...,The movie I'm thinking of is <|reserved_specia...,You might really enjoy <|reserved_special_toke...
498,<|begin_of_text|><|start_header_id|>system<|en...,"Well, considering your interest in comedy and ...","You know, if you’re in the mood for something ..."


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   prompt    500 non-null    object
 1   chosen    500 non-null    object
 2   rejected  500 non-null    object
dtypes: object(3)
memory usage: 11.8+ KB


###Duplicates

In [ ]:
print(df.duplicated().sum())

0


###Identical Chosen and Rejected Pairs

In [ ]:
identical_pairs = (df['chosen'] == df['rejected']).sum()
print(identical_pairs)

0


###Remove Hallucinated Labels/Prefixes

In [ ]:
prefix_triggers = ["rejected:", "response:", "assistant:", "rejected response:", "##", "###"]
hallucinated_prefixes = df['rejected'].str.lower().str.startswith(tuple(prefix_triggers)).sum()
hallucinated_prefixes += df['chosen'].str.lower().str.startswith(tuple(prefix_triggers)).sum()
print(hallucinated_prefixes)

14


In [ ]:
def strip_prefixes(text):
    #Strip unnecessary prefixes
    for prefix in ["rejected:", "response:", "assistant:", "rejected response:", "##", "###"]:
        if text.lower().startswith(prefix):
            text = text[len(prefix):].strip()
    #Strip trailing or leading markdown quotes
    if text.startswith(('"', "'")) and text.endswith(('"', "'")):
        text = text[1:-1].strip()
    return text

In [ ]:
df['rejected'] = df['rejected'].apply(strip_prefixes)
df['chosen'] = df['chosen'].apply(strip_prefixes)

###Remove Special Tokens

In [ ]:
#Remove the special token
def strip_special_tokens(text):
    if not isinstance(text, str):
        return ""
    return re.sub(SPECIAL_TOKEN_REGEX, "", text)

In [ ]:
df['chosen'] = df['chosen'].apply(strip_special_tokens)
df['rejected'] = df['rejected'].apply(strip_special_tokens)

###Remove Extra Whitespace

In [ ]:
#Collapse double spaces and strip leading/trailing edges
def collapse_whitespace(text):
    return " ".join(text.split()).strip()

In [ ]:
df['chosen'] = df['chosen'].apply(collapse_whitespace)
df['rejected'] = df['rejected'].apply(collapse_whitespace)

###Enforce Suffix Tokens

In [ ]:
#Append end-of-turn token if missing
def append_eot_suffix(text):
    if not text.endswith(EOT_TOKEN):
        return text + EOT_TOKEN
    return text

In [ ]:
df['chosen'] = df['chosen'].apply(append_eot_suffix)
df['rejected'] = df['rejected'].apply(append_eot_suffix)

###Sample Viewing

In [ ]:
sample = df.sample(1).iloc[0]
print(f"\n[PROMPT CONTEXT]:\n{sample['prompt']}")
print(f"\n[CHOSEN RESPONSE]:\n{sample['chosen']}")
print(f"\n[REJECTED RESPONSE]:\n{sample['rejected']}")


[PROMPT CONTEXT]:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a friendly and knowledgeable conversational recommender system designed to assist users in discovering TV shows, movies, or other content tailored to their preferences. Your goal is to provide personalized recommendations by understanding the user's tastes, preferences, and viewing history. You should engage in natural, conversational dialogue, ask clarifying questions when needed, and offer thoughtful suggestions that align with the user's interests.<|eot_id|><|start_header_id|>user<|end_header_id|>

I'm in the mood for a movie that's a mix of drama, romance, and comedy, with a unique storyline that's both entertaining and thought-provoking. I've enjoyed movies like Step Up Revolution and Alice in Wonderland in the past, which had a great balance of music, dance, and heartwarming moments. However, I'm looking for something a bit differe

##Save Final Dataset

In [ ]:
df.to_json(OUTPUT_FILE_CLEANED, orient="records", indent=4, force_ascii=False)